# Text Preprocessing Pipeline for NLP

This notebook walks through a standard **text preprocessing pipeline**: we take raw sentences and turn them into numeric vectors that machine learning models can use. The steps are:

1. **Lowercase** — so "Word" and "word" are treated the same  
2. **Remove punctuation** — keep only letters and spaces  
3. **Tokenize** — split text into individual words  
4. **Remove stopwords** — drop common words like "the", "is" that add little meaning  
5. **Lemmatize** — reduce words to base form (e.g. "running" → "run")  
6. **Vectorize** — convert the cleaned text to a bag-of-words count matrix  

We apply each step in order and inspect the result so you can see how the text changes.

## Setup

Run the cell below once to load libraries and download NLTK data (WordNet, stopwords, tokenizer). In Colab you may need to run it before the rest of the notebook.

In [6]:
# --- Setup: libraries and NLTK data ---
import string
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer

# One-time download of NLTK resources (WordNet for lemmatization, stopwords, tokenizer)
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package wordnet to /Users/Taj786/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/Taj786/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw to /Users/Taj786/nltk_data...
[nltk_data] Downloading package punkt to /Users/Taj786/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/Taj786/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## Sample data

We use a small dataframe of example sentences. Each step will add or update a column so we can compare raw input to the final vectors.

In [50]:
df = pd.DataFrame({
    'raw_text': [
        "This is a sample sentence, with punctuation!",
        "Another example: does it work?",
        "Lemmatization is important for NLP tasks.",
        "Tokenization breaks text into words.",
        "Vectorization converts text to numbers."
    ]
})
df

## Step 1: Lowercase

Normalize case so the same word in different forms (e.g. "NLP" vs "nlp") is treated as one token later.

In [52]:
df['processed_text'] = df['raw_text'].str.lower()
df[['raw_text', 'processed_text']]

,raw_text,processes_text
0,"This is a sample sentence, with punctuation!","this is a sample sentence, with punctuation!"
1,Another example: does it work?,another example: does it work?
2,Lemmatization is important for NLP tasks.,lemmatization is important for nlp tasks.
3,Tokenization breaks text into words.,tokenization breaks text into words.
4,Vectorization converts text to numbers.,vectorization converts text to numbers.


## Step 2: Remove punctuation

Strip punctuation so we only keep letters and spaces. This avoids tokens like "word." and "word" being treated as different. We use `string.punctuation` (e.g. !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~).

In [54]:
df['processed_text'] = df['processed_text'].str.translate(str.maketrans('', '', string.punctuation))
df[['raw_text', 'processed_text']]

## Step 3: Tokenize

Split each sentence into a list of words (tokens). This is required before we can filter stopwords or lemmatize.

In [70]:
df['tokens'] = df['processed_text'].apply(word_tokenize)
df[['processed_text', 'tokens']]

## Step 4: Remove stopwords

Drop common words (e.g. "the", "is", "a") that usually don't help with meaning. We use NLTK's English stopword list.

In [ ]:
english_stopwords = stopwords.words('english')
df['tokens_no_stop'] = df['tokens'].apply(
    lambda tokens: [t for t in tokens if t not in english_stopwords]
)
df[['tokens', 'tokens_no_stop']]

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

## Step 5: Lemmatize

Reduce words to their base form (e.g. "tasks" → "task", "breaks" → "break") so different forms count as the same feature. We use WordNet and `pos='v'` to prioritize verb forms.

In [73]:
lemmatizer = WordNetLemmatizer()
df['lemmatized_text'] = df['tokens_no_stop'].apply(
    lambda tokens: [lemmatizer.lemmatize(t, pos='v') for t in tokens]
)
df[['tokens_no_stop', 'lemmatized_text']]

,raw_text,processes_text,tokens,no_stop
0,"This is a sample sentence, with punctuation!",this is a sample sentence with punctuation,"[this, is, a, sample, sentence, with, punctuat...","[sample, sentence, punctuation]"
1,Another example: does it work?,another example does it work,"[another, example, does, it, work]","[another, example, work]"
2,Lemmatization is important for NLP tasks.,lemmatization is important for nlp tasks,"[lemmatization, is, important, for, nlp, tasks]","[lemmatization, important, nlp, tasks]"
3,Tokenization breaks text into words.,tokenization breaks text into words,"[tokenization, breaks, text, into, words]","[tokenization, breaks, text, words]"
4,Vectorization converts text to numbers.,vectorization converts text to numbers,"[vectorization, converts, text, to, numbers]","[vectorization, converts, text, numbers]"


## Step 6: Vectorize

Join the lemmatized tokens back into strings, then use **CountVectorizer** to build a bag-of-words matrix: each row is a document, each column is a word, values are counts.

In [86]:
df['lemmatized_joined'] = df['lemmatized_text'].apply(lambda x: ' '.join(x))

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['lemmatized_joined'])
df['vectorized'] = list(X.toarray())

# Show vocabulary (one column per word) and first document's counts
feature_names = vectorizer.get_feature_names_out()
print('Vocabulary size:', len(feature_names))
print('Sample words:', list(feature_names)[:12])
print('\nFirst row (document 0) counts:', df['vectorized'].iloc[0])

In [87]:
# Pipeline summary: raw input → cleaned text → vector
df[['raw_text', 'processed_text', 'lemmatized_joined']]

,raw_text,processes_text,tokens,no_stop,lemmatized_text
0,"This is a sample sentence, with punctuation!",this is a sample sentence with punctuation,"[this, is, a, sample, sentence, with, punctuat...","[sample, sentence, punctuation]","[sample, sentence, punctuation]"
1,Another example: does it work?,another example does it work,"[another, example, does, it, work]","[another, example, work]","[another, example, work]"
2,Lemmatization is important for NLP tasks.,lemmatization is important for nlp tasks,"[lemmatization, is, important, for, nlp, tasks]","[lemmatization, important, nlp, tasks]","[lemmatization, important, nlp, task]"
3,Tokenization breaks text into words.,tokenization breaks text into words,"[tokenization, breaks, text, into, words]","[tokenization, breaks, text, words]","[tokenization, break, text, word]"
4,Vectorization converts text to numbers.,vectorization converts text to numbers,"[vectorization, converts, text, to, numbers]","[vectorization, converts, text, numbers]","[vectorization, convert, text, number]"


In [92]:
# Optional: show full dataframe with all columns
df

**Pipeline complete.** The `vectorized` column holds the bag-of-words count vectors; use `X` (from the vectorize step) as the feature matrix for modeling.